# Drawing QA: commands

The main commands for the Drawing QA tool, one cell each. Run the cells top to bottom; change the settings cell to point at another package.

**Kernel:** pick this project's `.venv` as the notebook kernel (VS Code: *Select Kernel → Python Environments → .venv*). The first time, install everything from a terminal in the project folder:

```
python -m venv .venv
.venv\Scripts\python -m pip install -r requirements.txt
```

The tool only reports issues; it never changes a drawing.

## 1. Settings

In [ ]:
from pathlib import Path

PACKAGE = Path("input/Main")   # the package folder: PLC/, CableBlocks/, Terminations/, EquipmentList/
OUTPUT = None                  # None = reports/<package name>; or e.g. Path("reports/Main-2026-10-29") to keep a run
CONFIG = None                  # None = config/project.toml

PACKAGE.resolve()

## 2. Helper
Runs a `drawing_qa` command with this notebook's Python and prints what it says.

In [ ]:
import subprocess
import sys


def qa(*args):
    """python -m drawing_qa <args>, printed as it runs."""
    cmd = [sys.executable, "-m", "drawing_qa", *map(str, args)]
    if CONFIG is not None and args and args[0] in ("run", "survey"):
        cmd += ["--config", str(CONFIG)]
    print(">", " ".join(cmd[1:]))
    print()
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace") as p:
        for line in p.stdout:
            print(line, end="")
    if p.returncode:
        print()
        print(f"(exit code {p.returncode})")


def out_dir():
    return OUTPUT or Path("reports") / PACKAGE.resolve().name

## 3. Install or update the requirements
Only needed the first time, or after the dependencies change. Installs into the kernel's environment.

In [ ]:
%pip install -r requirements.txt

## 4. Survey a package
How many sheets can be read from their SHX text, need decoding (`--ocr`), are scans or are blank. Checks nothing; quick.

In [ ]:
qa("survey", "--input", PACKAGE)

## 5. Run the review
Reads every sheet, builds the claims, links them, traces every channel, runs the rules and writes the outputs.

In [ ]:
qa("run", "--input", PACKAGE, *(["--output", OUTPUT] if OUTPUT else []))

### Variations
Uncomment the one you need.
- `--exclude term`: leave an input type out of the review (`plc`, `cbd`, `term` or `equip`), e.g. termination drawings that are not up to date.
- `--ocr`: also read sheets plotted without SHX text by decoding their letter strokes.

In [ ]:
# qa("run", "--input", PACKAGE, "--exclude", "term")
# qa("run", "--input", PACKAGE, "--ocr")
# qa("run", "--input", PACKAGE, "--output", "reports/Main-2026-10-29")   # keep this run in its own folder

## 6. Open the outputs

In [ ]:
import os
import webbrowser

folder = out_dir()
for name in ("report.md", "QA_Register.xlsx", "system_map.html", "findings.json", "review.json"):
    path = folder / name
    print(f"{'ok ' if path.exists() else '-- '} {path}")

In [ ]:
webbrowser.open((out_dir() / "system_map.html").resolve().as_uri())   # the system map, in your browser

In [ ]:
os.startfile((out_dir() / "QA_Register.xlsx").resolve())   # the register, in Excel (Windows)

In [ ]:
from IPython.display import Markdown

Markdown((out_dir() / "report.md").read_text(encoding="utf-8"))   # the report, shown here

## 7. Run the tests
The tests make their own sample PDFs, so they need no client drawings.

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q"])

## 8. Help
Every option of each command.

In [ ]:
qa("run", "--help")

In [ ]:
qa("survey", "--help")